# For 1 example

In [1]:
!pip3 install -q transformers torch accelerate sentencepiece


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [5]:
import pandas as pd 
train_df = pd.read_csv('data/6/train.csv')
train_df

,sentiment,text,data_id
0,positive,Best sock ever made. Consistent quality...I hi...,6
1,positive,It always makes the water taste so good,6
2,positive,Looks good but did not need. Put into stock,6
3,negative,Unit worked perfectly for approximately 6 days...,6
4,positive,They fit perfectly on my ancient General Elect...,6
...,...,...,...
4909,negative,Bulky and slow. May change opinion in the summ...,6
4910,positive,The quality was good,6
4911,positive,"It's a filter, it was fine.",6
4912,positive,Love that it is so easy to use.,6


In [ ]:
# =========================
# 1-sample LLM classification pipeline
# =========================

import re
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------------------------
# 1. Config
# -------------------------
MODEL_NAME = "google/flan-t5-small"   # example open-source instruction model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Dataset labels
VALID_LABELS = train_df['sentiment'].unique().tolist()

# One sample only
sample_text = train_df['text'][0]
true_label = train_df['sentiment'][0]

# -------------------------
# 2. Prompt template
# -------------------------
def build_prompt(text: str) -> str:
    return f"""
        You are a text classification system.

        Task:
        Classify the given text into exactly one label from:
        {", ".join(VALID_LABELS)}

        Rules:
        - Return only one label
        - Do not explain
        - Do not write anything else

        Text:
        {text}

        Label:
        """.strip()

# -------------------------
# 3. Post-processing
# -------------------------
def normalize_label(raw_output: str, valid_labels: list[str]) -> str:
    text = raw_output.strip().lower()

    # exact match first
    if text in valid_labels:
        return text

    # first valid label appearing in output
    for label in valid_labels:
        if re.search(rf"\b{re.escape(label.lower())}\b", text):
            return label

    return "UNKNOWN"

# -------------------------
# 4. Load tokenizer/model
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

# For some causal models
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# -------------------------
# 5. Build prompt
# -------------------------
prompt = build_prompt(sample_text)
print("PROMPT:\n", prompt)

# -------------------------
# 6. Tokenize
# -------------------------
inputs = tokenizer( # Text -> Token IDs
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(DEVICE)

# -------------------------
# 7. Generate
# -------------------------
with torch.no_grad(): # no need to track gradients for inference (saves memory and computations and speeds up)
    outputs = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,      # deterministic for classification
        temperature=None,
        pad_token_id=tokenizer.eos_token_id
    )

# -------------------------
# 8. Decode only generated part
# -------------------------
raw_prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip() # Token IDs -> Text

# -------------------------
# 9. Post-process to final label
# -------------------------
pred_label = normalize_label(raw_prediction, VALID_LABELS)

# -------------------------
# 10. Compare
# -------------------------
print("\nRAW MODEL OUTPUT:", raw_prediction)
print("PREDICTED LABEL :", pred_label)
print("TRUE LABEL      :", true_label)
print("CORRECT?        :", pred_label == true_label)

PROMPT:
 You are a text classification system.

        Task:
        Classify the given text into exactly one label from:
        positive, negative

        Rules:
        - Return only one label
        - Do not explain
        - Do not write anything else

        Text:
        Best sock ever made. Consistent quality...I highly recommend Gold Toe  socks.

        Label:

RAW MODEL OUTPUT: positive
PREDICTED LABEL : positive
TRUE LABEL      : positive
CORRECT?        : True


# Functional

In [43]:
# ============================================
# LLM-based Sentiment Classification Pipeline
# Clean, reusable, assignment-friendly version
# ============================================

import re
from typing import List, Dict, Tuple

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# =========================================================
# Configuration
# =========================================================
MODEL_NAME = "google/flan-t5-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_INPUT_LENGTH = 512
MAX_NEW_TOKENS = 10


# =========================================================
# Prompt Templates
# =========================================================
PROMPT_TEMPLATES = {
    "prompt_1": """
    You are a text classification system.

    Classify the sentiment of the following text into exactly one label from:
    {labels}

    Return only one label.

    Text:
    {text}

    Label:
    """.strip(),

    "prompt_2": """
    Classify the sentiment of this text as one of these labels: {labels}.

    Text: {text}

    Answer with only one label.
    """.strip(),

        "prompt_3": """
    Read the text and predict its sentiment.

    Possible labels: {labels}

    Text:
    {text}

    Output only the correct label.
    """.strip(),

    "prompt_4": """
    You must perform sentiment classification.

    Allowed labels:
    {labels}

    Rules:
    - Choose one label only
    - Do not explain

    Text:
    {text}

    Sentiment:
    """.strip(),

    "prompt_5": """
    Determine whether the sentiment of the following text is:
    {labels}

    Text:
    {text}

    Return only the label.
    """.strip(),
    }


# =========================================================
# Model Loading
# =========================================================
def load_tokenizer(model_name: str):
    """Load and return the tokenizer for the selected model."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Some tokenizers may not define a pad token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return tokenizer


def load_model(model_name: str, device: str):
    """Load and return the seq2seq model on the target device."""
    dtype = torch.float16 if device == "cuda" else torch.float32

    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name,
        torch_dtype=dtype
    )
    model.to(device)
    model.eval()

    return model


# =========================================================
# Prompt Building
# =========================================================
def build_prompt(template: str, text: str, valid_labels: List[str]) -> str:
    """Create a formatted prompt for a single text sample."""
    label_text = ", ".join(valid_labels)
    return template.format(text=text, labels=label_text)


# =========================================================
# Text Preparation
# =========================================================
def prepare_model_inputs(
    tokenizer,
    prompt: str,
    device: str,
    max_input_length: int = MAX_INPUT_LENGTH
) -> Dict[str, torch.Tensor]:
    """Tokenize a prompt and move tensors to the target device."""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    )
    return {key: value.to(device) for key, value in inputs.items()}


# =========================================================
# Generation
# =========================================================
def generate_model_output(
    model,
    tokenizer,
    model_inputs: Dict[str, torch.Tensor],
    max_new_tokens: int = MAX_NEW_TOKENS
) -> str:
    """Generate raw text output from the model."""
    with torch.no_grad():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


# =========================================================
# Output Normalization
# =========================================================
def normalize_label(raw_output: str, valid_labels: List[str]) -> str:
    """
    Convert raw model output into one valid label.
    Returns 'UNKNOWN' if no valid label is detected.
    """
    cleaned_output = raw_output.strip().lower()
    normalized_labels = [label.lower() for label in valid_labels]

    # Exact match
    if cleaned_output in normalized_labels:
        return cleaned_output

    # Search for first valid label inside a longer response
    for label in normalized_labels:
        if re.search(rf"\b{re.escape(label)}\b", cleaned_output):
            return label

    return "UNKNOWN"


# =========================================================
# Single Prediction
# =========================================================
def predict_single_text(
    text: str,
    model,
    tokenizer,
    prompt_template: str,
    valid_labels: List[str],
    device: str
) -> Tuple[str, str, str]:
    """
    Predict sentiment for one text sample.

    Returns:
        prompt: formatted prompt
        raw_output: raw generated text
        predicted_label: normalized label
    """
    prompt = build_prompt(prompt_template, text, valid_labels)
    model_inputs = prepare_model_inputs(tokenizer, prompt, device)
    raw_output = generate_model_output(model, tokenizer, model_inputs)
    predicted_label = normalize_label(raw_output, valid_labels)

    return prompt, raw_output, predicted_label


# =========================================================
# Batch Prediction
# =========================================================
def predict_dataset(
    texts: List[str],
    model,
    tokenizer,
    prompt_template: str,
    valid_labels: List[str],
    device: str
) -> List[str]:
    """Predict labels for a list of texts using one prompt template."""
    predictions = []

    for text in texts:
        _, _, predicted_label = predict_single_text(
            text=text,
            model=model,
            tokenizer=tokenizer,
            prompt_template=prompt_template,
            valid_labels=valid_labels,
            device=device
        )
        predictions.append(predicted_label)

    return predictions


# =========================================================
# Multi-Prompt Prediction
# =========================================================
def predict_with_all_prompts(
    df: pd.DataFrame,
    text_column: str,
    model,
    tokenizer,
    prompt_templates: Dict[str, str],
    valid_labels: List[str],
    device: str
) -> pd.DataFrame:
    """
    Generate predictions for all prompt templates and
    return a copy of the dataframe with prediction columns added.
    """
    result_df = df.copy()

    for index, (_, template) in enumerate(prompt_templates.items(), start=1):
        column_name = f"out_label_Prompt_{index}"
        result_df[column_name] = predict_dataset(
            texts=result_df[text_column].tolist(),
            model=model,
            tokenizer=tokenizer,
            prompt_template=template,
            valid_labels=valid_labels,
            device=device
        )
        print(f"Finished predictions for {column_name}")

    return result_df


# =========================================================
# Example: Run Pipeline for One Sample
# =========================================================
def run_single_sample_demo(train_df: pd.DataFrame) -> None:
    """Run the full pipeline for one sample and print outputs."""
    valid_labels = sorted(train_df["sentiment"].astype(str).str.lower().unique().tolist())
    sample_text = str(train_df.loc[0, "text"])
    true_label = str(train_df.loc[0, "sentiment"]).lower()

    tokenizer = load_tokenizer(MODEL_NAME)
    model = load_model(MODEL_NAME, DEVICE)

    prompt, raw_output, predicted_label = predict_single_text(
        text=sample_text,
        model=model,
        tokenizer=tokenizer,
        prompt_template=PROMPT_TEMPLATES["prompt_1"],
        valid_labels=valid_labels,
        device=DEVICE
    )

    print("PROMPT:\n")
    print(prompt)
    print("\nRAW MODEL OUTPUT:", repr(raw_output))
    print("PREDICTED LABEL :", predicted_label)
    print("TRUE LABEL      :", true_label)
    print("CORRECT?        :", predicted_label == true_label)


# =========================================================
# Example: Run Pipeline for Whole Dataset
# =========================================================
def run_full_prompt_inference(
    input_df: pd.DataFrame,
    text_column: str = "text",
    label_column: str = "sentiment"
) -> pd.DataFrame:
    """
    Run all prompt templates on the full dataframe.
    Returns a dataframe containing prediction columns.
    """
    valid_labels = sorted(input_df[label_column].astype(str).str.lower().unique().tolist())

    tokenizer = load_tokenizer(MODEL_NAME)
    model = load_model(MODEL_NAME, DEVICE)

    output_df = predict_with_all_prompts(
        df=input_df,
        text_column=text_column,
        model=model,
        tokenizer=tokenizer,
        prompt_templates=PROMPT_TEMPLATES,
        valid_labels=valid_labels,
        device=DEVICE
    )

    return output_df


# =========================================================
# Usage
# =========================================================
# Assumes train_df already exists and has:
# - train_df["text"]
# - train_df["sentiment"]

# 1) Run one-sample demo
# run_single_sample_demo(train_df)

# 2) Run full prompt inference
# predicted_df = run_full_prompt_inference(train_df)

# 3) Check prediction columns
# print(predicted_df.head())

In [ ]:
train_df.head(100)

,sentiment,text,data_id
0,positive,Best sock ever made. Consistent quality...I hi...,6
1,positive,It always makes the water taste so good,6
2,positive,Looks good but did not need. Put into stock,6
3,negative,Unit worked perfectly for approximately 6 days...,6
4,positive,They fit perfectly on my ancient General Elect...,6
5,positive,Grat value and work the same as the brand name...,6
6,negative,The brand new drum roller seized up already. ...,6
7,positive,The factory filter is very expensive. This doe...,6
8,negative,DOES NOT SIT EVENLY ON STOVE. DOES NOT LOOK LI...,6
9,positive,Looks great in the kitchen!,6


In [ ]:
run_single_sample_demo(train_df)

PROMPT:

You are a text classification system.

    Classify the sentiment of the following text into exactly one label from:
    negative, positive

    Return only one label.

    Text:
    Best sock ever made. Consistent quality...I highly recommend Gold Toe  socks.

    Label:

RAW MODEL OUTPUT: 'positive'
PREDICTED LABEL : positive
TRUE LABEL      : positive
CORRECT?        : True


In [ ]:
# predicted_df = run_full_prompt_inference(train_df.head(1000))

Finished predictions for out_label_Prompt_1
Finished predictions for out_label_Prompt_2
Finished predictions for out_label_Prompt_3
Finished predictions for out_label_Prompt_4
Finished predictions for out_label_Prompt_5


In [54]:
predicted_df

,sentiment,text,data_id,out_label_Prompt_1,out_label_Prompt_2,out_label_Prompt_3,out_label_Prompt_4,out_label_Prompt_5
0,positive,Best sock ever made. Consistent quality...I hi...,6,positive,positive,positive,positive,positive
1,positive,It always makes the water taste so good,6,positive,positive,positive,positive,positive
2,positive,Looks good but did not need. Put into stock,6,negative,negative,negative,negative,negative
3,negative,Unit worked perfectly for approximately 6 days...,6,negative,negative,negative,negative,negative
4,positive,They fit perfectly on my ancient General Elect...,6,positive,positive,positive,positive,positive
...,...,...,...,...,...,...,...,...
995,positive,I use two years and very pleased. It works cl...,6,positive,positive,positive,positive,positive
996,positive,Great product and easy installation.,6,positive,positive,positive,positive,positive
997,positive,This product works better than the one that ca...,6,positive,positive,positive,positive,positive
998,positive,It fit great and seller even sent us complimen...,6,positive,positive,positive,positive,positive


In [56]:
accuracy = (predicted_df['sentiment'] == predicted_df['out_label_Prompt_1']).mean()
print(f"Accuracy: {accuracy:.2%}")

accuracy2 = (predicted_df['sentiment'] == predicted_df['out_label_Prompt_2']).mean()
print(f"Accuracy Prompt 2: {accuracy2:.2%}")   

accuracy3 = (predicted_df['sentiment'] == predicted_df['out_label_Prompt_3']).mean()
print(f"Accuracy Prompt 3: {accuracy3:.2%}")   

accuracy4 = (predicted_df['sentiment'] == predicted_df['out_label_Prompt_4']).mean()
print(f"Accuracy Prompt 4: {accuracy4:.2%}")    

accuracy5 = (predicted_df['sentiment'] == predicted_df['out_label_Prompt_5']).mean()
print(f"Accuracy Prompt 5: {accuracy5:.2%}")

Accuracy: 85.50%
Accuracy Prompt 2: 82.40%
Accuracy Prompt 3: 75.80%
Accuracy Prompt 4: 81.40%
Accuracy Prompt 5: 63.90%


In [57]:
def predict_dataset_single_prompt(
    df: pd.DataFrame,
    text_column: str,
    model,
    tokenizer,
    prompt_template: str,
    valid_labels: List[str],
    device: str
) -> List[str]:
    """
    Generate predictions for the entire dataset using ONE prompt.

    Returns:
        List[str]: predicted labels for each row
    """

    predictions = []

    for text in df[text_column]:
        # 1. Build prompt
        prompt = build_prompt(prompt_template, str(text), valid_labels)

        # 2. Tokenize
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # 3. Generate output
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        # 4. Decode
        raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

        # 5. Normalize label
        pred_label = normalize_label(raw_output, valid_labels)

        predictions.append(pred_label)

    return predictions

preds = predict_dataset_single_prompt(
    df=train_df,
    text_column="text",
    model=model,
    tokenizer=tokenizer,
    prompt_template=PROMPT_TEMPLATES["prompt_1"],
    valid_labels=VALID_LABELS,
    device=DEVICE
)

In [60]:
train_df['predicted_sentiment'] = preds
train_df.head()

,sentiment,text,data_id,predicted_sentiment
0,positive,Best sock ever made. Consistent quality...I hi...,6,positive
1,positive,It always makes the water taste so good,6,positive
2,positive,Looks good but did not need. Put into stock,6,negative
3,negative,Unit worked perfectly for approximately 6 days...,6,negative
4,positive,They fit perfectly on my ancient General Elect...,6,positive


In [61]:
accuracy_single_prompt = (train_df['sentiment'] == train_df['predicted_sentiment']).mean()
print(f"Accuracy with single prompt: {accuracy_single_prompt:.2%}")

Accuracy with single prompt: 86.47%
